# Import records.csv and map to student_list array


In [ ]:
# import record into student list array to use (no pandas)
import csv
import io
import os
import requests

# Team size configuration: allows 4–10 with safe defaults
MIN_TEAM_SIZE = 4
MAX_TEAM_SIZE = 10
DEFAULT_TEAM_SIZE = 5

# check if input is integer
def _parse_int(s):
    try:
        return int(s)
    except Exception:
        return None

def get_team_size():
    try:
        raw = input(
            f"Enter team size ({MIN_TEAM_SIZE}-{MAX_TEAM_SIZE}) [{DEFAULT_TEAM_SIZE}]: "
        ).strip()
        ts = _parse_int(raw) if raw else DEFAULT_TEAM_SIZE
        if MIN_TEAM_SIZE <= ts <= MAX_TEAM_SIZE:
            return ts
        else:
            print(f"Invalid team size; using default {DEFAULT_TEAM_SIZE}.")
    except Exception:
        pass

    return DEFAULT_TEAM_SIZE

TEAM_SIZE = get_team_size()
print(f"Using team size: {TEAM_SIZE}")

In [ ]:
# read raw data from Github
GITHUB_CSV_URL = "https://raw.githubusercontent.com/lekekbot/ntu-cs/main/SC1003/group%20project/records.csv"

student_list = []

resp = requests.get(GITHUB_CSV_URL, timeout=30)
resp.raise_for_status()  # will error clearly if the URL is bad

csv_stream = io.StringIO(resp.content.decode("utf-8-sig"))

reader = csv.DictReader(csv_stream)

for row in reader:
    # be a bit defensive about CGPA conversion
    cgpa_str = (row.get("CGPA") or "").strip()
    try:
        cgpa_val = float(cgpa_str) if cgpa_str else None
    except ValueError:
        cgpa_val = None

    student = {
        "tutorial":   (row.get("Tutorial Group") or "").strip(),
        "student_id": (row.get("Student ID") or "").strip(),
        "name":       (row.get("Name") or "").strip(),
        "school":     (row.get("School") or "").strip(),
        "gender":     (row.get("Gender") or "").strip(),
        "cgpa":       cgpa_val,
    }
    student_list.append(student)

# quick sanity check
print(f"Loaded {len(student_list)} students. First record:")
print(student_list[0] if student_list else "No rows found.")

# Loop into student_list to sort them based on their tutorial group
the sorted group will be a dictionary of array
```
{
  "G-1": [
    {
      'tutorial': 'G-1',
      'student_id': '5002',
      'name': 'Aarav Singh',
      'school': 'CCDS',
      'gender': 'Male',
      'cgpa': 4.02
    },
    ........
  ],
  "G-2": [
    .....
  ]
}
```






In [ ]:
tutorial_grp = {}

for student in student_list:
    tutorial = student["tutorial"]
    if tutorial not in tutorial_grp:
        tutorial_grp[tutorial] = []
    tutorial_grp[tutorial].append(student)

print(tutorial_grp)

# Create Helpers

To help ease the duplication of codes used in the algorithm, a set of helper functions are created.  
These functions handle team attribute counting, limit checking, and concentration evaluation, making the main algorithm cleaner and easier to maintain.

### Overview

- **`max_allowed_per_attribute(team_size)`**  
  Determines the maximum number of members allowed per attribute (e.g., gender) within a team.  
  Since the team size can be odd, the formula `(team_size + 1) // 2` ensures a fair distribution.  
  For example, for a team size of 5, the best ratio would be **3:2**.

- **`counts_in_team(team, key)`**  
  Builds a dictionary that accumulates the count of each attribute value (e.g., gender or nationality) within a team.  
  This helps in tracking how balanced the team composition is.

- **`violates_cap_after_add(team, student, key, team_size)`**  
  Checks whether adding a given student would cause the team to exceed the maximum allowed number for that attribute.  
  Returns `True` if the cap would be violated, otherwise `False`.

- **`concentration_after_add(team, student, key)`**  
  Calculates the projected count of an attribute after adding a student to the team.  
  Useful for evaluating how concentrated a particular attribute (e.g., same gender or same major) would become after the addition.




In [ ]:
def max_allowed_per_attribute(team_size):
    return (team_size + 1) // 2

def counts_in_team(team, key):
    counts_dict = {}
    for student in team:
        attribute_value = student[key]
        counts_dict[attribute_value] = counts_dict.get(attribute_value, 0) + 1
    return counts_dict

def violates_cap_after_add(team, student, key, team_size):
    max_allowed = max_allowed_per_attribute(team_size)
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1 > max_allowed

def concentration_after_add(team, student, key):
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1

## Placement Scoring

`placement_score()` calculates how suitable it is to add a student to a team.  
Lower scores mean better fits.

- **Hard caps:** Block placements that exceed school/gender limits.  
- **Soft penalties:** Add small costs for high school or gender concentration.  
- **CGPA balance:** Keep team average close to tutorial mean.  
- **Fill bias:** Slightly favor smaller teams.


In [ ]:
def placement_score(team, student, tutorial_mean_t, team_size):
    HARD = 10_000  # block if it creates a cap violation
    SOFT_SCHOOL = 5  # gentle penalty for concentration
    SOFT_GENDER = 3
    SOFT_FILL = 1  # slightly prefer smaller teams now

    # hard caps: do not place if it would exceed allowed count
    if violates_cap_after_add(team, student, "school", team_size):
        return HARD
    if violates_cap_after_add(team, student, "gender", team_size):
        return HARD

    # soft penalties: encourage spreading out attributes
    school_conc = concentration_after_add(team, student, "school")
    gender_conc = concentration_after_add(team, student, "gender")

    # CGPA balance (via t_score): keep team avg close to tutorial mean
    new_sum = sum(m["t_score"] for m in team) + student["t_score"]
    new_size = len(team) + 1
    cgpa_penalty = abs(new_sum / new_size - tutorial_mean_t)

    # fill penalty: slightly prefer teams with fewer members
    fill_penalty = len(team) * SOFT_FILL

    return (
        (SOFT_SCHOOL * school_conc)
        + (SOFT_GENDER * gender_conc)
        + cgpa_penalty
        + fill_penalty
    )

# Build Teams

The `build_teams()` function automatically forms balanced and diverse student teams within a tutorial group.  
It mixes students of different performance levels and backgrounds while keeping each team’s overall ability close to the class average.

### Steps Overview

1. **Sort by T-Score**  
   Students are first sorted by their `t_score` (academic performance + minority in schools/gender in tutorial group) in ascending order.

2. **Create "Snake" Order**  
   To mix strong and weak students, the algorithm alternates between the highest and lowest scores, producing an order like:  
   `[highest, lowest, 2nd highest, 2nd lowest, ...]`  
   This ensures early placement of both top and bottom performers.

3. **Initialize Teams**  
   Calculates how many teams are needed (based on `team_size`) and creates empty team lists.

4. **Greedy Placement**  
   Iterates through students in the snake order:
   - For each student, tests every available team using `placement_score()`.  
   - Chooses the team with the lowest (best) placement score.  
   - If all teams are full, places the student in the smallest team as a fallback.

5. **Label & Finalize Teams**  
   After forming all teams:
   - Each team gets a unique ID (`tutorial_id-T1`, `tutorial_id-T2`, etc.).  
   - Computes each team’s average `t_score` and attaches it to members for reference.

### Notes
- `placement_score()` controls how good each placement is by considering:
  - Diversity (school, gender)
  - Academic balance (`t_score` closeness to mean)
  - Team fill preference

In [ ]:
def build_teams(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):
    # sort students by T score (ascending)
    students_sorted = sorted(students, key=lambda x: x["t_score"])

    # snake order: [highest, lowest, 2nd highest, 2nd lowest, ...]
    snake = []
    L = 0
    R = len(students_sorted) - 1
    while L <= R:
        snake.append(students_sorted[R]); R -= 1
        if L <= R:
            snake.append(students_sorted[L]); L += 1

    # correct number of teams (ceil division)
    num_of_teams = (len(students) + team_size - 1) // team_size
    teams = [[] for _ in range(num_of_teams)]

    # get the mean of a tutorial class
    tutorial_mean_t = sum(s["t_score"] for s in students) / len(students)

    # evaluate all available teams and calculate the placement scores to find the best fit
    for student in snake:
        best_team_index = None
        best_placement_score = float("inf")

        # enumerate to return both index and team
        for team_index, current_team in enumerate(teams):
            if len(current_team) >= team_size:
                continue
            score = placement_score(current_team, student, tutorial_mean_t, team_size)
            if score < best_placement_score:
                best_placement_score = score
                best_team_index = team_index

        if best_team_index is None:
            # safety fallback, pick smallest team
            best_team_index = min(range(len(teams)), key=lambda k: len(teams[k]))
        teams[best_team_index].append(student)

    # drop any accidental empties (extra safety)
    teams = [t for t in teams if t]

    # label teams and provide average tscore for each teams
    labeled = []
    for i, team in enumerate(teams, 1):
        team_id = f"{tutorial_id}-T{i}"
        team_mean = sum(x["t_score"] for x in team) / len(team)
        for x in team:
            x["team_id"] = team_id
            x["team_tscore"] = round(team_mean, 3)
        labeled.append(team)

    return labeled


## Repair Teams

The `repair_teams()` function performs a minimal, targeted balance. It finds the two teams with means farthest above and below the tutorial mean, evaluates all 1-for-1 swaps between them, and applies the single swap that most reduces total deviation, while respecting **team size**, **school caps**, and **gender caps**.

### How It Works

1. **Identify Extremes**  
   Compute each team’s mean and deviation from the tutorial mean; pick the highest and lowest.

2. **Evaluate Swaps**  
   Try all 1-for-1 swaps between the two extreme teams; skip any swap that would violate school/gender caps.

3. **Commit Best Swap**  
   Apply the swap that yields the largest reduction in combined deviation. If none improves, do nothing.

4. **Update Team Stats**  
   Refresh `team_tscore` for the two affected teams; keep existing `team_id`s.

### Summary

- Reduces the largest mean gap with a single safe swap  
- Preserves team sizes and respects caps  
- Simple and non-iterative; no full reassignment

In [ ]:
def repair_teams(teams, team_size, tutorial_mean_t, threshold=0.1):

    def team_mean(team):
        return sum(s["t_score"] for s in team) / len(team)

    def deviation(team):
        return team_mean(team) - tutorial_mean_t

    cap = max_allowed_per_attribute(team_size)

    # check if 1-for-1 swap preserves school/gender caps
    def cap_ok_after_swap(team_a, team_b, student_a, student_b):
        for attribute in ("school", "gender"):
            a_counts = counts_in_team(team_a, attribute).copy()
            b_counts = counts_in_team(team_b, attribute).copy()

            # remove high and low of students in a and b
            a_counts[student_a[attribute]] = a_counts.get(student_a[attribute], 0) - 1
            b_counts[student_b[attribute]] = b_counts.get(student_b[attribute], 0) - 1

            # add high and low of students in a and b (swapping)
            a_counts[student_b[attribute]] = a_counts.get(student_b[attribute], 0) + 1
            b_counts[student_a[attribute]] = b_counts.get(student_a[attribute], 0) + 1

            if any(v > cap for v in a_counts.values()) or any(v > cap for v in b_counts.values()):
                return False
        return True

    # repair teams whose mean deviates beyond +-threshold
    # limit iterations to avoid infinite loops (if difference is always 0.1+)

    # prevent iteration from going unlimited
    max_iterations = len(teams) * 2
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        highs = [t for t in teams if deviation(t) > threshold]
        lows = [t for t in teams if deviation(t) < -threshold]

        if not highs or not lows:
            break

        best_pair = None
        best_cost = None

        for hi in highs:
            hi_mean = team_mean(hi)
            hi_size = len(hi)
            for lo in lows:
                lo_mean = team_mean(lo)
                lo_size = len(lo)
                base_cost = abs(hi_mean - tutorial_mean_t) + abs(lo_mean - tutorial_mean_t)

                for i, s_hi in enumerate(hi):
                    for j, s_lo in enumerate(lo):
                        if not cap_ok_after_swap(hi, lo, s_hi, s_lo):
                            continue

                        new_hi_mean = hi_mean + (s_lo["t_score"] - s_hi["t_score"]) / hi_size
                        new_lo_mean = lo_mean + (s_hi["t_score"] - s_lo["t_score"]) / lo_size
                        new_cost = abs(new_hi_mean - tutorial_mean_t) + abs(new_lo_mean - tutorial_mean_t)

                        # Require improvement and prefer largest reduction
                        if new_cost < base_cost:
                            if best_cost is None or new_cost < best_cost:
                                best_cost = new_cost
                                best_pair = (hi, lo, i, j, new_hi_mean, new_lo_mean, new_cost)

        if best_pair is None:
            # stop if no improvememts from changes
            break

        # Commit the swap
        hi, lo, i, j, new_hi_mean, new_lo_mean, _ = best_pair
        hi[i], lo[j] = lo[j], hi[i]

        # Update stored team_tscore for affected teams
        new_hi = team_mean(hi)
        new_lo = team_mean(lo)
        for m in hi:
            m["team_tscore"] = round(new_hi, 3)
        for m in lo:
            m["team_tscore"] = round(new_lo, 3)

    return teams

## Create Groups by T-Score

The `create_groups_by_tscore()` function builds diverse, balanced teams for a tutorial.  
It combines **academic normalization**, **diversity bonuses**, **placement scoring**, and a **repair phase** to optimize team composition.

### Workflow Overview

1. **Normalize CGPA → Compute T-Scores**  
   - Each student’s CGPA is scaled between 0 and 1 based on the tutorial’s min/max range.  
   - Diversity bonuses are added:  
     - +0.2 if from the least represented gender  
     - +0.1 if from the least represented school  
   - The resulting `t_score` reflects both ability and diversity value.

2. **Compute Tutorial Mean**  
   Calculates the average `t_score` across all students to use as a balancing reference.

3. **Initial Team Formation**  
   Calls `build_teams()` to assign students into teams using the diversity-aware placement algorithm.  
   Teams are built to respect size and attribute caps.

4. **Repair Phase**  
   Calls `repair_teams()` to rebalance teams in a single pass, refining mean balance and reducing cap violations.

5. **Summary Output**  
   Prints overall and per-team stats:
   - Tutorial mean T-score  
   - Average team deviation from the mean  
   - Each team’s ID, mean T-score, and deviation

### Key Notes
- `t_score` = normalized CGPA + diversity bonuses (bonus given to minority)
- The algorithm ensures teams are **diverse**, **balanced**, and **close to the overall mean performance**.
- Ideally, the target should not exceed > 0.1 for each tutorial group against the tutorial means

In [ ]:
def create_groups_by_tscore(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):

    # min/max CGPA for normalization
    cgpas = [student["cgpa"] for student in students]

    min_cgpa = min(cgpas)
    max_cgpa = max(cgpas)
    cgpa_range = max_cgpa - min_cgpa

    # counts in tutorial
    gender_counts = {}
    school_counts = {}
    for student in students:
        gender_counts[student["gender"]] = gender_counts.get(student["gender"], 0) + 1
        school_counts[student["school"]] = school_counts.get(student["school"], 0) + 1

    min_gender_count = min(gender_counts.values()) if gender_counts else 0
    min_school_count = min(school_counts.values()) if school_counts else 0

    # compute t-scores for each students
    total_tscore = 0
    for student in students:
        normalized_cgpa = (student["cgpa"] - min_cgpa) / cgpa_range
        gender_diversity_bonus = (
            0.2 if gender_counts[student["gender"]] == min_gender_count else 0.0
        )  # 0.2 bonus if gender is the least in a tutorial
        school_diversity_bonus = (
            0.1 if school_counts[student["school"]] == min_school_count else 0.0
        )  # +0.1 if school count is the least in a tutorial
        t_score = normalized_cgpa + gender_diversity_bonus + school_diversity_bonus

        student["t_score"] = round(t_score, 3)
        total_tscore += student["t_score"]

    # tutorial mean T-score
    mean_tscore = total_tscore / len(students)

    # build initial teams with diversity-aware placement
    teams = build_teams(tutorial_id, students, team_size)

    # repair teams to reduce violations and polish CGPA balance
    teams = repair_teams(teams, team_size, mean_tscore)

    # summary printout
    team_mean_diffs = [
        abs(sum(student["t_score"] for student in t) / len(t) - mean_tscore)
        for t in teams
    ]
    avg_team_diff = sum(team_mean_diffs) / len(team_mean_diffs)

    print(f"\n=== {tutorial_id} summary ===")
    print(f"Tutorial mean T-score: {mean_tscore:.3f}")
    print(f"Actual average team diff from tutorial mean: {avg_team_diff:.4f}")

    for team in teams:
        team_mean = sum(s["t_score"] for s in team) / len(team)
        diff = abs(team_mean - mean_tscore)
        print(
            f"{tutorial_id} | {team[0]['team_id']} | Team mean T={team_mean:.3f} | Diff={diff:.3f}"
        )

    return teams

In [ ]:
# run algorithm
sorted_tutorial_grp = {}

for tutorial_id, students in tutorial_grp.items():
    teams = create_groups_by_tscore(tutorial_id, students, TEAM_SIZE)
    sorted_tutorial_grp[tutorial_id] = teams

In [ ]:
# export to csv
output_data = []

for tutorial_id, teams in sorted_tutorial_grp.items():
    for team in teams:
        for student in team:
            row = {
                "Tutorial Group": student["tutorial"],
                "Student ID": student["student_id"],
                "School": student["school"],
                "Name": student["name"],
                "Gender": student["gender"],
                "CGPA": student["cgpa"],
                "Team Assigned": student.get("team_id", f"{tutorial_id}-Unknown"),
            }
            output_data.append(row)

output_file = "./output.csv"
fieldnames = [
    "Tutorial Group",
    "Student ID",
    "School",
    "Name",
    "Gender",
    "CGPA",
    "Team Assigned",
]

with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_data)

print(f"\nTeam assignments exported to {output_file}")
print(f"Total students processed: {len(output_data)}")
print(
    f"Total teams created: {sum(len(teams) for teams in sorted_tutorial_grp.values())}"
)

In [ ]:
import matplotlib.pyplot as plt

# helpers
def _find_field_name(sample_row, candidates):
    lower_map = {k.lower(): k for k in sample_row.keys()}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _teams_from_rows(rows):
    teams = {}
    for r in rows:
        tid = (
            r.get("Team Assigned")
            or r.get("team assigned")
            or r.get("team")
            or r.get("Team")
        )
        if tid is None:
            continue
        teams.setdefault(str(tid), []).append(r)
    return teams  # dict: {team_id: [rows...]}


def _to_float(val):
    if val is None:
        return None
    s = str(val).strip().replace(",", "")
    if not s:
        return None
    try:
        return float(s)
    except:
        return None


def _gender_code(val):
    """Return 'M' or 'F' if clearly identifiable, else None."""
    if val is None:
        return None
    s = str(val).strip().lower()
    if not s:
        return None
    if s[0] == "m":
        return "M"
    if s[0] == "f":
        return "F"
    return None


# gender-mix distribution chart
def plot_gender_mix_distribution(team_data_rows, team_size_hint=None):
    if not team_data_rows:
        print("No data to plot (gender mix).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (gender mix).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    gender_col = _find_field_name(any_row, ["Gender"])
    if gender_col is None:
        print("Couldn't find a 'Gender' column.")
        return

    from collections import Counter, defaultdict

    comp_counts_by_size = defaultdict(Counter)  # {size: Counter("xM+yF" -> n)}
    size_counts = Counter()

    for members in teams_dict.values():
        m = f = 0
        for r in members:
            g = _gender_code(r.get(gender_col))
            if g == "M":
                m += 1
            elif g == "F":
                f += 1
        size = len(members)
        if team_size_hint is not None:
            # lock bins to a fixed size if desired
            if m + f != team_size_hint:
                # if blanks/others exist, force F from hint
                f = max(0, team_size_hint - m)
            size = team_size_hint
        else:
            size = m + f  # only count rows recognized as M/F

        label = f"{m}M+{f}F"
        comp_counts_by_size[size][label] += 1
        size_counts[size] += 1

    if not size_counts:
        print("No recognizable M/F data to plot.")
        return

    primary_size = max(size_counts.items(), key=lambda kv: kv[1])[0]

    def _plot_size(size, counts):
        labels = [f"{m}M+{size-m}F" for m in range(size, -1, -1)]
        values = [counts.get(lbl, 0) for lbl in labels]

        fig = plt.figure(figsize=(8, 4.5))
        ax = fig.gca()
        ax.bar(range(len(labels)), values)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_xlabel("Team gender composition")
        ax.set_ylabel("Number of teams")
        ax.set_title(f"Gender Mix Distribution (Team Size {size})")
        for x, v in enumerate(values):
            if v > 0:
                ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
        plt.tight_layout()
        plt.show()

    # plot the most common team size first
    _plot_size(primary_size, comp_counts_by_size[primary_size])

    # plot other sizes if present
    for size, counts in comp_counts_by_size.items():
        if size == primary_size:
            continue
        _plot_size(size, counts)


# teams by number of distinct schools chart
def plot_school_count_distribution(team_data_rows):
    if not team_data_rows:
        print("No data to plot (schools).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (schools).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    school_col = _find_field_name(
        any_row, ["School"]
    )
    if school_col is None:
        print("Couldn't find a 'School' column.")
        return

    from collections import Counter

    counter = Counter()
    max_sch = 0

    for members in teams_dict.values():
        schools = []
        for r in members:
            val = r.get(school_col)
            val = "" if val is None else str(val).strip()
            if val:
                schools.append(val)
        distinct = len(set(schools))
        counter[distinct] += 1
        if distinct > max_sch:
            max_sch = distinct

    xs = list(range(1, max_sch + 1)) if max_sch > 0 else [0]
    ys = [counter.get(x, 0) for x in xs]

    fig = plt.figure(figsize=(8, 4.5))
    ax = fig.gca()
    ax.bar(range(len(xs)), ys)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([str(x) for x in xs])
    ax.set_xlabel("Number of distinct schools within a team")
    ax.set_ylabel("Number of teams")
    ax.set_title("School Diversity: Teams by Count of Distinct Schools")
    for x, v in enumerate(ys):
        if v > 0:
            ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()


# average CGPA per team with overall line chart
def plot_average_cgpa_per_team(team_data_rows, show_hist=True):
    if not team_data_rows:
        print("No data to plot (CGPA).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (CGPA).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    cgpa_col = _find_field_name(any_row, ["CGPA"])
    if cgpa_col is None:
        print("Couldn't find a 'CGPA' column.")
        return

    team_ids, team_avgs, all_cgpas = [], [], []

    for tid, members in sorted(teams_dict.items(), key=lambda kv: kv[0]):
        vals = []
        for r in members:
            v = _to_float(r.get(cgpa_col))
            if v is not None:
                vals.append(v)
                all_cgpas.append(v)
        if vals:
            avg = sum(vals) / len(vals)
            team_ids.append(str(tid))
            team_avgs.append(avg)

    overall_avg = (sum(all_cgpas) / len(all_cgpas)) if all_cgpas else 0.0

    fig2 = plt.figure(figsize=(7.5, 4.5))
    ax2 = fig2.gca()
    bins = max(5, min(12, len(team_avgs)))
    ax2.hist(team_avgs, bins=bins)
    ax2.set_xlabel("Team average CGPA")
    ax2.set_ylabel("Number of teams")
    ax2.set_title("Distribution of Team Average CGPA")
    ax2.axvline(overall_avg, color="red", linewidth=2)
    ymax = ax2.get_ylim()[1]
    ax2.text(
        overall_avg,
        ymax * 0.95,
        f"{overall_avg:.2f}",
        ha="left",
        va="top",
        fontsize=9,
        color="red",
    )
    plt.tight_layout()
    plt.show()


# (adjust team_size_hint if teams are fixed size, e.g., 5)
plot_gender_mix_distribution(output_data, team_size_hint=TEAM_SIZE)
plot_school_count_distribution(output_data)
plot_average_cgpa_per_team(output_data, show_hist=True)